# Assignment 8
Extract the clusters representing customer groups based on their credit card spending data.
Consider the possibility that clusters may exist in different subspaces. For this purpose follow the
following steps:
 - a. Download the Credit Card Data from the www.kaggle.com
 - b. Apply the PROCLUS clustering techniques for finding clusters prevailing in different
subspaces.
- c. Find the cluster quality in terms of Dunn Index and DB index

## Loading Dataset

In [42]:
import pandas as pd
import numpy as np
df = pd.read_csv('/kaggle/input/ccdata/CC GENERAL.csv')
df.sample(10)

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
2995,C13084,5030.927831,1.000000,99.00,99.00,0.00,2999.769235,0.166667,0.166667,0.000000,0.583333,10,2,6000.0,1144.381323,1143.067027,0.000000,12
5484,C15638,1638.425340,1.000000,0.00,0.00,0.00,37.921481,0.000000,0.000000,0.000000,0.083333,1,0,1800.0,459.151425,335.813771,0.000000,12
8684,C18921,1929.555023,1.000000,621.11,621.11,0.00,6691.286753,0.333333,0.333333,0.000000,0.666667,29,5,6000.0,2406.905963,549.027080,0.000000,12
5404,C15557,1906.283349,1.000000,4016.42,1058.18,2958.24,0.000000,1.000000,0.583333,1.000000,0.000000,0,35,8000.0,1344.622375,526.831031,0.000000,12
7110,C17303,990.780585,1.000000,844.45,844.45,0.00,0.000000,0.416667,0.416667,0.000000,0.000000,0,6,3000.0,530.463101,227.905412,0.000000,12
48,C10050,229.867179,1.000000,2390.60,1402.93,987.67,0.000000,1.000000,0.666667,1.000000,0.000000,0,87,3300.0,2543.953559,175.657825,0.916667,12
5699,C15858,126.169985,1.000000,1155.00,0.00,1155.00,0.000000,0.833333,0.000000,0.833333,0.000000,0,10,3900.0,1375.353007,158.020499,0.909091,12
5683,C15841,92.902071,1.000000,812.74,618.40,194.34,0.000000,1.000000,1.000000,0.416667,0.000000,0,18,4000.0,708.651818,176.003434,0.583333,12
7292,C17489,95.055337,1.000000,133.45,16.80,116.65,437.390028,0.500000,0.083333,0.333333,0.083333,1,6,2500.0,469.271786,160.662631,0.090909,12
605,C10630,202.190447,0.545455,2011.81,2011.81,0.00,0.000000,0.416667,0.416667,0.000000,0.000000,0,8,2100.0,3144.544042,165.070133,0.800000,12


In [43]:
df = df.sample(n=400, random_state=42)

### Selecting Features and Scaling the Data

In [44]:
from sklearn.preprocessing import StandardScaler

# Select spending-related features
features = ['BALANCE', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE']
data = df[features].copy()

# Standardize the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

In [45]:
data.sample(5)

,BALANCE,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE
6696,812.839893,1160.09,932.09,228.00,3129.703318
8506,1087.565213,0.00,0.00,0.00,216.514140
5706,106.390419,1038.38,1038.38,0.00,0.000000
2087,1681.162870,1138.85,0.00,1138.85,4044.377495
6984,51.651430,398.98,0.00,398.98,0.000000


In [46]:
data = data_scaled

### Helper functions

In [47]:
def euclidean_distance(point1, point2):
    return np.sqrt(np.sum((point1 - point2) ** 2))

In [48]:
def assign_points_full_space(data, medoids):
    """
    Assign each data point to the closest medoid using full space.
    Returns a list of cluster assignments (list of lists of indices).
    """
    clusters = {i: [] for i in range(len(medoids))}
    for idx, point in enumerate(data):
        distances = [euclidean_distance(point, medoids[i]) for i in range(len(medoids))]
        assigned = np.argmin(distances)
        clusters[assigned].append(idx)
    return clusters

In [49]:
def compute_cluster_variances(data, indices):
    """
    For a given cluster (indices), compute the variance along each dimension.
    """
    if len(indices) == 0:
        return None
    cluster_data = data[indices]
    return np.var(cluster_data, axis=0)

In [50]:
def select_subspace_dims(variances, num_dims):
    """
    Select the indices of the 'num_dims' dimensions with lowest variance.
    """
    return np.argsort(variances)[:num_dims]

In [51]:
def assign_points_subspace(data, medoid_indices, subspaces):
    """
    Reassign points to clusters using only the selected subspace for each cluster.
    medoid_indices: list of indices of the medoid for each cluster
    subspaces: list of lists/arrays where each entry contains the dimension indices selected for that cluster.
    Returns new cluster assignment (as a dict).
    """
    clusters = {i: [] for i in range(len(medoid_indices))}
    for idx, point in enumerate(data):
        # Compute distance in each cluster's subspace
        distances = []
        for cluster in range(len(medoid_indices)):
            dims = subspaces[cluster]
            # Use the point’s coordinates in the chosen dimensions
            medoid_point = data[medoid_indices[cluster]][dims]
            distance = euclidean_distance(point[dims], medoid_point)
            distances.append(distance)
        assigned = np.argmin(distances)
        clusters[assigned].append(idx)
    return clusters

In [52]:
def update_medoid(data, indices, dims):
    """
    For a given cluster (a list of indices) and the chosen subspace dimensions,
    select a new medoid: the point that minimizes the sum of distances to others (using only the subspace dims).
    """
    if len(indices) == 0:
        return None
    best_idx = indices[0]
    best_dist = np.inf
    for i in indices:
        # Sum distances from point i to all other points in the cluster (using selected dims)
        distances = [euclidean_distance(data[i][dims], data[j][dims]) for j in indices]
        total = np.sum(distances)
        if total < best_dist:
            best_dist = total
            best_idx = i
    return best_idx

## Config

In [53]:
# Number of clusters
k = 3

# Number of dimensions to select (for each cluster)
subspace_dims = 2

## Simplefied Implementation of Proclus clustering

In [54]:
# ---------------------------
# 3. Simplified PROCLUS Algorithm (Iterative Version)
# ---------------------------

# 3.1: Initial medoids: choose fixed indices (here we choose index 0 and index 1)
initial_medoid_indices = [0, 1, 5]
medoid_indices = initial_medoid_indices.copy()

# Initial assignment: using full space (could also be done in subspace after selection)
clusters = assign_points_full_space(data, [data[i] for i in medoid_indices])
print("Initial clusters (full space):", clusters)

# Run for 10 iterations
for itr in range(5):
    print("\nIteration", itr + 1)
    
    # Step 2: For each cluster, identify the subspace by selecting dimensions with the lowest variance.
    selected_dims = {}
    for cluster_idx, indices in clusters.items():
        variances = compute_cluster_variances(data, indices)
        if variances is not None:
            dims = select_subspace_dims(variances, subspace_dims)
            selected_dims[cluster_idx] = dims
        else:
            selected_dims[cluster_idx] = np.array([], dtype=int)
    
    print("Selected subspaces for each cluster:", selected_dims)
    
    # Step 3: Reassign points using the distances computed in the selected subspace.
    clusters = assign_points_subspace(data, medoid_indices, [selected_dims[i] for i in range(k)])
    print("Clusters after reassignment in subspaces:", clusters)
    
    # Step 4: Update medoids based on current clusters and their corresponding subspaces.
    new_medoid_indices = []
    for cluster_idx, indices in clusters.items():
        dims = selected_dims[cluster_idx]
        new_medoid = update_medoid(data, indices, dims)
        if new_medoid is None:
            # If a cluster is empty, retain the previous medoid.
            new_medoid = medoid_indices[cluster_idx]
        new_medoid_indices.append(new_medoid)
    
    print("Updated medoid indices:", new_medoid_indices)
    
    # Check for convergence (optional): If medoid indices did not change, you might break early.
    if new_medoid_indices == medoid_indices:
        print("Convergence reached.")
        break
    else:
        medoid_indices = new_medoid_indices.copy()

print("\nFinal clusters:", clusters)
print("Final medoid indices:", medoid_indices)

Initial clusters (full space): {0: [0, 19, 21, 23, 26, 31, 32, 33, 34, 39, 51, 62, 65, 68, 76, 84, 93, 95, 102, 105, 107, 112, 117, 120, 123, 127, 128, 129, 130, 132, 145, 148, 152, 157, 159, 161, 167, 168, 185, 191, 193, 195, 201, 211, 216, 222, 226, 227, 229, 231, 233, 234, 235, 237, 246, 251, 254, 268, 271, 274, 276, 282, 289, 301, 305, 306, 308, 315, 321, 322, 331, 335, 340, 362, 366, 372, 374, 376, 378, 391, 397, 398], 1: [1, 3, 4, 6, 9, 10, 11, 12, 15, 16, 17, 18, 24, 25, 28, 29, 30, 35, 37, 42, 43, 45, 46, 47, 48, 52, 53, 54, 56, 57, 59, 61, 64, 66, 67, 69, 70, 71, 73, 74, 77, 78, 79, 81, 82, 85, 86, 87, 88, 89, 91, 97, 98, 99, 101, 104, 106, 108, 110, 118, 121, 124, 125, 131, 133, 134, 137, 138, 139, 140, 141, 143, 147, 149, 150, 154, 155, 156, 163, 164, 165, 166, 169, 170, 173, 179, 181, 183, 184, 186, 187, 188, 190, 194, 196, 198, 207, 210, 213, 215, 218, 219, 220, 221, 223, 228, 230, 239, 241, 244, 245, 247, 249, 250, 252, 253, 257, 259, 260, 261, 262, 264, 267, 272, 273, 27

##  Evaluation Metrics: Dunn Index and Davies-Bouldin Index

In [55]:
def intra_cluster_diameter(data, indices):
    """
    Compute the diameter (the maximum pairwise distance) of the cluster.
    """
    max_dist = 0
    for i in indices:
        for j in indices:
            dist = euclidean_distance(data[i], data[j])
            if dist > max_dist:
                max_dist = dist
    return max_dist

def inter_cluster_distance(data, indices1, indices2):
    """
    Compute the minimum distance between any two points from two clusters.
    """
    min_dist = np.inf
    for i in indices1:
        for j in indices2:
            dist = euclidean_distance(data[i], data[j])
            if dist < min_dist:
                min_dist = dist
    return min_dist

def compute_dunn_index(data, clusters):
    """
    The Dunn index is defined as the ratio between the minimum inter-cluster distance 
    and the maximum intra-cluster diameter. Higher values indicate better clustering.
    """
    # Compute max intra-cluster diameter
    intra_diameters = []
    for indices in clusters.values():
        if len(indices) > 0:
            diam = intra_cluster_diameter(data, indices)
            intra_diameters.append(diam)
    max_intra = max(intra_diameters) if intra_diameters else 0
    
    # Compute minimum inter-cluster distance (pairwise between clusters)
    cluster_keys = list(clusters.keys())
    inter_dists = []
    for i in range(len(cluster_keys)):
        for j in range(i+1, len(cluster_keys)):
            d = inter_cluster_distance(data, clusters[cluster_keys[i]], clusters[cluster_keys[j]])
            inter_dists.append(d)
    min_inter = min(inter_dists) if inter_dists else 0
    
    dunn = min_inter / max_intra if max_intra > 0 else 0
    return dunn

def compute_db_index(data, clusters):
    """
    The Davies-Bouldin index is the average similarity measure of each cluster with its most similar cluster.
    Lower values indicate better clustering.
    """
    cluster_keys = list(clusters.keys())
    # Compute intra-cluster dispersion for each cluster (average distance to the medoid)
    dispersions = {}
    for key, indices in clusters.items():
        if len(indices) == 0:
            dispersions[key] = 0
        else:
            # For dispersion we can use average distance between all pairs in the cluster.
            sum_dists = 0
            count = 0
            for i in indices:
                for j in indices:
                    if i != j:
                        sum_dists += euclidean_distance(data[i], data[j])
                        count += 1
            dispersions[key] = sum_dists / count if count > 0 else 0

    # Compute DB index for each cluster
    db_ratios = []
    for i in cluster_keys:
        max_ratio = -np.inf
        for j in cluster_keys:
            if i == j:
                continue
            # Use the distance between cluster centroids (here we use medoids as proxies)
            # For simplicity, calculate using all dimensions
            if len(clusters[i]) > 0 and len(clusters[j]) > 0:
                # Here we calculate the distance between the medoids
                medoid_i = data[update_medoid(data, clusters[i], np.arange(data.shape[1]))]
                medoid_j = data[update_medoid(data, clusters[j], np.arange(data.shape[1]))]
                centroid_dist = euclidean_distance(medoid_i, medoid_j)
                ratio = (dispersions[i] + dispersions[j]) / centroid_dist if centroid_dist > 0 else np.inf
                if ratio > max_ratio:
                    max_ratio = ratio
        db_ratios.append(max_ratio)
    
    db_index = np.mean(db_ratios)
    return db_index

In [56]:
new_clusters = clusters
dunn_index = compute_dunn_index(data, new_clusters)
db_index = compute_db_index(data, new_clusters)

print("\nDunn index:", dunn_index)
print("Davies-Bouldin index:", db_index)


Dunn index: 0.002324721236854941
Davies-Bouldin index: 4.418717247772768
